In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import os

DATA_DIR = "../data"
PLOTS_DIR = "../plots"

THEORETICAL_STRIKE_ZONE = {
    'plate_x_min': -0.83,
    'plate_x_max': 0.83,
    'plate_z_min': 1.5,
    'plate_z_max': 3.5
}

In [2]:
def load_data():
    """Load and filter statcast data (called pitches only)"""
    dfs = []
    for year in [2023, 2024, 2025]:
        path = os.path.join(DATA_DIR, f"statcast_{year}.parquet")
        df = pd.read_parquet(path)
        df['year'] = year
        dfs.append(df)
    
    combined = pd.concat(dfs, ignore_index=True)
    combined = combined.dropna(subset=['plate_x', 'plate_z', 'type', 'description'])
    combined = combined[combined['type'].isin(['S', 'B'])]
    
    swung_descriptions = [
        'foul', 'hit_into_play', 'swinging_strike', 'foul_tip', 
        'swinging_strike_blocked', 'foul_bunt', 'missed_bunt', 'bunt_foul_tip'
    ]
    combined = combined[~combined['description'].isin(swung_descriptions)]
    
    combined['is_strike'] = (combined['type'] == 'S').astype(int)
    return combined

df = load_data()
print(f"Loaded {len(df):,} called pitches total")

Loaded 1,136,983 called pitches total


In [3]:
def calculate_strike_probability(df, bins=25):
    """Calculate strike probability by location bin"""
    x_bins = np.linspace(-1.5, 1.5, bins + 1)
    z_bins = np.linspace(0, 5, bins + 1)
    
    df = df.copy()
    df['x_bin'] = pd.cut(df['plate_x'], bins=x_bins, labels=False, include_lowest=True)
    df['z_bin'] = pd.cut(df['plate_z'], bins=z_bins, labels=False, include_lowest=True)
    
    df = df.dropna(subset=['x_bin', 'z_bin'])
    df['x_bin'] = df['x_bin'].astype(int)
    df['z_bin'] = df['z_bin'].astype(int)
    
    prob = df.groupby(['z_bin', 'x_bin']).agg(
        strike_prob=('is_strike', 'mean'),
        count=('is_strike', 'count')
    ).reset_index()
    
    prob_matrix = prob.pivot(index='z_bin', columns='x_bin', values='strike_prob')
    count_matrix = prob.pivot(index='z_bin', columns='x_bin', values='count')
    
    prob_matrix = prob_matrix.sort_index(ascending=False)
    count_matrix = count_matrix.sort_index(ascending=False)
    
    prob_matrix = prob_matrix.values
    count_matrix = count_matrix.values
    
    return prob_matrix, count_matrix, x_bins, z_bins

In [4]:
pitch_type_names = {
    'FF': 'Four-Seam Fastball',
    'SI': 'Sinker',
    'FC': 'Cutter',
    'SL': 'Slider',
    'CH': 'Changeup',
    'CU': 'Curveball',
    'KC': 'Knuckle Curve'
}

df = df[df['pitch_type'].isin(pitch_type_names.keys())]
print(f"After filtering to main pitch types: {len(df):,} pitches")

pitch_types = list(pitch_type_names.keys())

pitch_type_colors = {
    'FF': '#E41A1C',  
    'SI': '#FF7F00',  
    'FC': '#FFFF33',  
    'SL': '#A65628',  
    'CH': '#4DAF4A',  
    'CU': '#377EB8',  
    'KC': '#984EA3',  
}

colors = [pitch_type_colors[pt] for pt in pitch_types]

print("Pitch types to plot:")
for pt in pitch_types:
    count = len(df[df['pitch_type'] == pt])
    print(f"  {pt} ({pitch_type_names[pt]}): {count:,} pitches")

In [5]:
X_LIMITS = (-1.25, 1.25)
Y_LIMITS = (1.25, 3.75)

MIN_COUNT = 20

print("Inset placement parameters defined.")
print(f"Main plot limits: X={X_LIMITS}, Y={Y_LIMITS}")

In [6]:
def plot_pitch_type_zone_size_grid(distance_results, sz):
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    import numpy as np

    def fmt_ft_in(ft):
        """1.0401 ft  →  1' 0.48\""""
        total_in = ft * 12
        feet     = int(total_in // 12)
        inches   = total_in - feet * 12
        return f"{feet}' {inches:.2f}\""

    def fmt_diff_in(ft):
        """+0.0181 ft  →  +0.22\""""
        inches = ft * 12
        return f"{'+' if inches >= 0 else ''}{inches:.2f}\""

    all_dist = next(d for label, _, d in distance_results if label == "All")

    lookup = {}
    for label, n, d in distance_results:
        if label == "All":
            continue
        lookup[label] = (d, d - all_dist, n)

    sorted_pitch_types = sorted(lookup.keys(), key=lambda x: lookup[x][0])

    n_cols = 4
    n_rows = 2

    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(11, 5.5),
        gridspec_kw=dict(hspace=0.20, wspace=0.15)
    )
    axes = axes.flatten()

    cmap = mcolors.LinearSegmentedColormap.from_list(
        "sz_div", ["#378ADD", "#f0f0f0", "#D85A30"]
    )
    norm = mcolors.TwoSlopeNorm(vmin=-0.10, vcenter=0.0, vmax=0.10)

    for idx, ax in enumerate(axes):
        ax.set_xticks([])
        ax.set_yticks([])

        if idx < len(sorted_pitch_types):
            pitch_type = sorted_pitch_types[idx]
            avg_d, vs_all, n = lookup[pitch_type]
            color = cmap(norm(vs_all))

            for spine in ax.spines.values():
                spine.set_linewidth(1.5)
                spine.set_edgecolor(
                    "#C04010" if vs_all >  0.015 else
                    "#1860A0" if vs_all < -0.015 else
                    "#aaaaaa"
                )

            ax.set_facecolor(color)

            diff_color = (
                "#7A2A08" if vs_all >  0.015 else
                "#083A6E" if vs_all < -0.015 else
                "#555555"
            )

            ax.text(0.5, 0.85, pitch_type,
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=12, fontweight='bold', color='#333333')

            ax.text(0.5, 0.60, pitch_type_names.get(pitch_type, ''),
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=8, color='#444444')

            ax.text(0.5, 0.35, fmt_ft_in(avg_d),
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=11, color='#222222')

            ax.text(0.5, 0.10, fmt_diff_in(vs_all),
                    transform=ax.transAxes, ha='center', va='center',
                    fontsize=11, fontweight='bold', color=diff_color)
        else:
            ax.set_visible(False)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, orientation='vertical',
                        fraction=0.02, pad=0.02, aspect=25)
    cbar.set_label('Δ avg dist vs all (inches)', fontsize=11)
    tick_ft = [-0.10, -0.05, 0.00, 0.05, 0.10]
    cbar.set_ticks(tick_ft)
    cbar.set_ticklabels([fmt_diff_in(t) for t in tick_ft])

    sz_cx = (sz['plate_x_min'] + sz['plate_x_max']) / 2
    sz_cz = (sz['plate_z_min'] + sz['plate_z_max']) / 2
    fig.suptitle(
        f"Perceived strike zone size by pitch type\n"
        f"Avg 50% contour distance from zone center "
        f"(x={sz_cx:.2f}, z={sz_cz:.2f} ft)  ·  baseline = {fmt_ft_in(all_dist)}",
        fontsize=12, y=1.02
    )

    plt.savefig(os.path.join(PLOTS_DIR, "pitch_type_zone_size_grid.png"),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: pitch_type_zone_size_grid.png")

In [7]:
def create_all_contours_plot(df, x_bins, z_bins, min_count=20):
    """Create the main contour plot with zoom panels along each strike zone edge"""
    import matplotlib.gridspec as gridspec

    contour_bins = 30
    buf = 0.12
    sz = THEORETICAL_STRIKE_ZONE

    prob_all, count_all, _, _ = calculate_strike_probability(df, bins=100)
    prob_all_filtered = np.where(count_all >= min_count, prob_all, np.nan)

    prob_contour, count_contour, x_edges_c, z_edges_c = calculate_strike_probability(df, bins=contour_bins)
    prob_contour_filtered = np.where(count_contour >= min_count, prob_contour, np.nan)
    x_centers_c = x_edges_c[:-1] + (x_edges_c[1] - x_edges_c[0]) / 2
    z_centers_c = z_edges_c[:-1] + (z_edges_c[1] - z_edges_c[0]) / 2
    Xc, Zc = np.meshgrid(x_centers_c, z_centers_c)

    pitch_contour_data = []
    for idx, pitch_type in enumerate(pitch_types):
        df_pt = df[df['pitch_type'] == pitch_type]
        if len(df_pt) < 50:
            continue
        prob_c, count_c, x_e2, z_e2 = calculate_strike_probability(df_pt, bins=contour_bins)
        prob_c_f = np.where(count_c >= min_count, prob_c, np.nan)
        xc2 = x_e2[:-1] + (x_e2[1] - x_e2[0]) / 2
        zc2 = z_e2[:-1] + (z_e2[1] - z_e2[0]) / 2
        Xc2, Zc2 = np.meshgrid(xc2, zc2)
        pitch_contour_data.append((pitch_type, Xc2, Zc2, prob_c_f))

    def extract_contour_paths(X, Z, prob):
        fig_t, ax_t = plt.subplots()
        paths = []
        try:
            cs = ax_t.contour(X, Z, prob, levels=[0.5])
            paths = [p.vertices for p in cs.get_paths() if len(p.vertices)]
        except Exception:
            pass
        plt.close(fig_t)
        return paths

    all_paths = extract_contour_paths(Xc, Zc, prob_contour_filtered)
    for (pitch_type, Xc2, Zc2, prob_c_f) in pitch_contour_data:
        all_paths.extend(extract_contour_paths(Xc2, Zc2, prob_c_f))

    if all_paths:
        all_pts = np.vstack(all_paths)
        all_cx, all_cz = all_pts[:, 0], all_pts[:, 1]
    else:
        all_cx = np.array([sz['plate_x_min'], sz['plate_x_max']])
        all_cz = np.array([sz['plate_z_min'], sz['plate_z_max']])

    sz_cx = (sz['plate_x_min'] + sz['plate_x_max']) / 2
    sz_cz = (sz['plate_z_min'] + sz['plate_z_max']) / 2

    def arc_length_weighted_mean_dist(paths, cx, cz):
        total_weight = 0.0
        total_weighted_dist = 0.0
        for verts in paths:
            if len(verts) < 2:
                d = np.hypot(verts[0, 0] - cx, verts[0, 1] - cz)
                total_weighted_dist += d
                total_weight += 1.0
                continue
            segs_start = verts[:-1]
            segs_end   = verts[1:]
            midpoints  = (segs_start + segs_end) / 2.0
            seg_lens   = np.hypot(segs_end[:, 0] - segs_start[:, 0],
                                  segs_end[:, 1] - segs_start[:, 1])
            dists      = np.hypot(midpoints[:, 0] - cx,
                                  midpoints[:, 1] - cz)
            total_weighted_dist += np.dot(seg_lens, dists)
            total_weight        += seg_lens.sum()

        return total_weighted_dist / total_weight if total_weight > 0 else np.nan

    distance_results = []

    overall_paths = extract_contour_paths(Xc, Zc, prob_contour_filtered)
    overall_dist  = arc_length_weighted_mean_dist(overall_paths, sz_cx, sz_cz)
    distance_results.append(("All", len(df), overall_dist))

    for (pitch_type, Xc2, Zc2, prob_c_f) in pitch_contour_data:
        df_pt = df[df['pitch_type'] == pitch_type]
        count_paths = extract_contour_paths(Xc2, Zc2, prob_c_f)
        mean_dist   = arc_length_weighted_mean_dist(count_paths, sz_cx, sz_cz)
        distance_results.append((pitch_type, len(df_pt), mean_dist))

    half_w = (sz['plate_x_max'] - sz['plate_x_min']) / 2
    half_h = (sz['plate_z_max'] - sz['plate_z_min']) / 2
    corner_dist = np.hypot(half_w, half_h)

    print("\n" + "=" * 58)
    print(f"  Strike Zone Center: x={sz_cx:.3f} ft,  z={sz_cz:.3f} ft")
    print(f"  Zone half-width={half_w:.3f} ft | half-height={half_h:.3f} ft")
    print(f"  Center→corner (ref): {corner_dist:.3f} ft")
    print("=" * 58)
    print(f"  {'Pitch Type':<18}  {'N Pitches':>10}  {'Avg Dist (ft)':>14}  {'vs All':>9}")
    print("-" * 58)
    
    plot_pitch_type_zone_size_grid(distance_results, sz)

    all_dist_ref = distance_results[0][2]
    for label, n, d in distance_results:
        if np.isnan(d):
            d_str    = "    N/A"
            diff_str = "    N/A"
        else:
            d_str    = f"  {d:.4f}"
            diff     = d - all_dist_ref
            diff_str = f"  {diff:+.4f}"
        print(f"  {label:<18}  {n:>10,}  {d_str:>14}  {diff_str:>9}")

    print("=" * 58 + "\n")

    def edge_zoom(vals, thresh_fn, fallback_center):
        near = vals[thresh_fn(vals)]
        if len(near):
            return (near.min() - buf, near.max() + buf)
        return (fallback_center - buf * 2, fallback_center + buf * 2)

    top_ylim   = edge_zoom(all_cz, lambda v: v >  sz['plate_z_max'] - 0.5, sz['plate_z_max'])
    bot_ylim   = edge_zoom(all_cz, lambda v: v <  sz['plate_z_min'] + 0.5, sz['plate_z_min'])
    left_xlim  = edge_zoom(all_cx, lambda v: v <  sz['plate_x_min'] + 0.5, sz['plate_x_min'])
    right_xlim = edge_zoom(all_cx, lambda v: v >  sz['plate_x_max'] - 0.5, sz['plate_x_max'])

    fig = plt.figure(figsize=(18, 16))
    gs = gridspec.GridSpec(
        3, 3,
        height_ratios=[1, 5, 1],
        width_ratios=[1, 5, 1],
        hspace=0.08, wspace=0.08
    )
    ax_main  = fig.add_subplot(gs[1, 1])
    ax_top   = fig.add_subplot(gs[0, 1])
    ax_bot   = fig.add_subplot(gs[2, 1])
    ax_left  = fig.add_subplot(gs[1, 0])
    ax_right = fig.add_subplot(gs[1, 2])

    def draw_on(ax):
        im = ax.imshow(
            prob_all_filtered,
            extent=[x_bins[0], x_bins[-1], z_bins[0], z_bins[-1]],
            origin='lower', cmap='bwr', vmin=0, vmax=1,
            aspect='auto', alpha=0.5
        )
        ax.add_patch(Rectangle(
            (sz['plate_x_min'], sz['plate_z_min']),
            sz['plate_x_max'] - sz['plate_x_min'],
            sz['plate_z_max'] - sz['plate_z_min'],
            linewidth=2, edgecolor='lime', facecolor='none'
        ))
        try:
            ax.contour(Xc, Zc, prob_contour_filtered,
                       levels=[0.5], colors=['black'], linewidths=3.5, linestyles='solid')
        except Exception as e:
            print("Main contour failed:", e)
        for (pitch_type, Xc2, Zc2, prob_c_f) in pitch_contour_data:
            try:
                ax.contour(Xc2, Zc2, prob_c_f,
                           levels=[0.5], colors=[pitch_type_colors[pitch_type]], linewidths=3.5, linestyles='dashed')
            except Exception:
                pass
        return im

    im = draw_on(ax_main)
    ax_main.set_xlim(X_LIMITS)
    ax_main.set_ylim(Y_LIMITS)
    ax_main.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax_main.axhline(2.5, color='gray', linestyle='--', alpha=0.5)
    ax_main.tick_params(axis='both', which='both', bottom=False, left=False,
                        labelbottom=False, labelleft=False)

    draw_on(ax_top)
    ax_top.set_xlim(X_LIMITS)
    ax_top.set_ylim(3.2, top_ylim[1])
    ax_top.set_ylabel('Z (ft)', fontsize=15)
    ax_top.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    ax_top.tick_params(axis='y', labelsize=12)
    ax_top.set_title('Top Edge ↑', fontsize=20, pad=3)

    draw_on(ax_bot)
    ax_bot.set_xlim(X_LIMITS)
    ax_bot.set_ylim(bot_ylim[0], 1.75)
    ax_bot.set_ylabel('Z (ft)', fontsize=15)
    ax_bot.set_xlabel('Plate X (ft)', fontsize=15)
    ax_bot.tick_params(axis='both', labelsize=12)
    ax_bot.set_title('↓ Bottom Edge', fontsize=20, pad=3)

    draw_on(ax_left)
    ax_left.set_xlim(left_xlim[0], -0.7)
    ax_left.set_ylim(Y_LIMITS)
    ax_left.set_xlabel('X (ft)', fontsize=15)
    ax_left.set_ylabel('Plate Z (ft)', fontsize=15)
    ax_left.tick_params(axis='both', labelsize=12)
    ax_left.set_title('←\nLeft\nEdge', fontsize=20, pad=3)

    draw_on(ax_right)
    ax_right.set_xlim(0.7, right_xlim[1])
    ax_right.set_ylim(Y_LIMITS)
    ax_right.set_xlabel('X (ft)', fontsize=15)
    ax_right.tick_params(axis='x', labelsize=12)
    ax_right.tick_params(axis='y', labelleft=False, labelsize=12)
    ax_right.set_title('→\nRight\nEdge', fontsize=20, pad=3)

    main_legend_elements = [
        Rectangle((0, 0), 1, 1, facecolor='none', edgecolor='lime', linewidth=2,
                  label='Approx "True" Zone'),
        Line2D([0], [0], color='black', linewidth=2.5, label='All Pitches'),
    ]
    pitch_type_legend_elements = []
    for (pitch_type, _, _, _) in pitch_contour_data:
        pitch_type_legend_elements.append(
            Line2D([0], [0], color=pitch_type_colors[pitch_type], linestyle='dashed', linewidth=2,
                   label=f'{pitch_type} ({pitch_type_names[pitch_type]})')
        )

    main_leg = ax_main.legend(
        handles=main_legend_elements,
        loc='upper center',
        fontsize=15,
        ncol=2,
    )
    ax_main.add_artist(main_leg)

    ax_main.legend(
        handles=pitch_type_legend_elements,
        loc='center',
        fontsize=13,
        ncol=2,
        title='Pitch Type',
        title_fontsize=12,
    )

    cbar_ax = fig.add_axes([0.15, 0.05, 0.70, 0.018])
    cbar = fig.colorbar(im, cax=cbar_ax, orientation='horizontal')
    cbar.set_label('Strike %', fontsize=20)
    cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
    cbar.set_ticklabels(['0%', '25%', '50%', '75%', '100%'])
    cbar.ax.tick_params(labelsize=10)

    top_ymin    = 3.35
    top_ymax    = 3.6
    bottom_ymin = 1.4
    bottom_ymax = 1.65
    left_xmin   = -0.95
    left_xmax   = -0.8
    right_xmin  = 0.80
    right_xmax  = 1.0

    ax_top.set_ylim(top_ymin, top_ymax)
    ax_bot.set_ylim(bottom_ymin, bottom_ymax)
    ax_left.set_xlim(left_xmin, left_xmax)
    ax_right.set_xlim(right_xmin, right_xmax)

    plt.savefig(os.path.join(PLOTS_DIR, "strike_probability_all_contours_pitch_type.png"),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: strike_probability_all_contours_pitch_type.png")

In [8]:
x_bins = np.linspace(-1.5, 1.5, 26)
z_bins = np.linspace(0, 5, 26)

create_all_contours_plot(df, x_bins, z_bins, min_count=1)